# Urban Safety — Classification: ML Models

Trains Logistic Regression, Decision Tree, and Random Forest classifiers to predict street-level safety class from built-environment features.

**Pre-ML analysis in `01_London_ClassificationML_FeatureInspection.ipynb`.**
**Feature extraction in `00_OSM_Data_Fetch.ipynb`.**

## Cell 1 — Imports and config

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sns.set(rc={'figure.figsize': (8, 8)})
pd.set_option('display.float_format', '{:.3f}'.format)

CRIME_FEATURES_CSV = 'csvFiles/segment_crime_scores_w-id.csv'
FEATURE_COLS = ['lighting', 'visibility', 'connectivity', 'enclosure']

## Cell 2 — Load pre-labelled segment features
Loads `segment_crime_scores_w-id.csv` produced by `01_London_ClassificationML_FeatureInspection.ipynb`.

In [ ]:
features = pd.read_csv(CRIME_FEATURES_CSV)
print(f'\u2713 Loaded {len(features):,} segments')
print(f'  Columns: {list(features.columns)}')
print(f'\nSafety class distribution:')
print(features['safety_class'].value_counts().sort_index())

## Cell 3 — Model training
Trains Logistic Regression, Decision Tree, and Random Forest on the combined multi-borough dataset.

In [ ]:
data = features[FEATURE_COLS + ['safety_class']].dropna(subset=['safety_class']).copy()
X = data[FEATURE_COLS]
y = data['safety_class']

print(f'Training data shape: X = {X.shape}, y = {y.shape}')
print(f'Class distribution:\n{y.value_counts()}\n')

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=FEATURE_COLS, index=X.index)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train set: {X_train.shape[0]:,} samples')
print(f'Test set:  {X_test.shape[0]:,} samples\n')

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print(f'Logistic Regression accuracy: {(lr_pred == y_test).mean():.3f}\n')
print(classification_report(y_test, lr_pred))

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
print(f'Decision Tree accuracy: {(dt_pred == y_test).mean():.3f}\n')
print(classification_report(y_test, dt_pred))

rf = RandomForestClassifier(n_estimators=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print(f'Random Forest (5 trees) accuracy: {(rf_pred == y_test).mean():.3f}\n')
print(classification_report(y_test, rf_pred))

## Cell 4 — Confusion matrices

In [ ]:
classes = ['safe', 'low', 'medium', 'high']
os.makedirs('plots', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, pred, title in zip(
    axes, [lr_pred, dt_pred, rf_pred],
    ['Logistic Regression', 'Decision Tree', 'Random Forest (5 trees)']
):
    cm = confusion_matrix(y_test, pred, labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                ax=ax, linewidths=0.5, cbar=False)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion matrices - London safety classification', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 5 — Feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model, title in zip(axes, [dt, rf], ['Decision Tree', 'Random Forest (5 trees)']):
    importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values()
    colors = ['#F44336' if v == importances.max() else 'steelblue' for v in importances.values]
    ax.barh(importances.index, importances.values, color=colors, edgecolor='none')
    ax.set_title(f'Feature importance - {title}', fontsize=12)
    ax.set_xlabel('Gini importance')
    ax.spines[['top', 'right']].set_visible(False)
plt.suptitle('Feature importance - London safety classification (multi-borough)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusions

The models were trained on a combined dataset of **~36,000 street segments across 5 London boroughs**
(Westminster, Islington, Hackney, Tower Hamlets, Southwark), with features derived from OSM geometry
and Mapillary street lamp detections.

**Accuracy summary:**

| Model | Accuracy |
|---|---|
| Logistic Regression | ~47.9% |
| Decision Tree | ~79.2% |
| Random Forest (5 trees) | ~73.1% |

The large gap between Logistic Regression and the tree models indicates the morphology-crime relationship
is **non-linear**. Decision Tree achieves the best performance, suggesting the safety class boundaries are
well-captured by axis-aligned splits on the feature space.

**Enclosure** (building footprint coverage ratio x street canyon ratio H/W) is the dominant predictor,
aligning with **Routine Activity Theory** (Cohen & Felson, 1979).

**Lighting** (Mapillary lamp density) contributes less than expected, likely due to patchy Mapillary coverage.